# 09 — Add-on Capabilities: Barcodes, Query Fields & High-Res Mode

**Time**: ~10 minutes · **Prebuilt models with add-ons** · **No training required**

---

## Insurance Scenario

Insurance documents often contain elements beyond standard text and tables:

- **Barcodes/QR codes** on claim forms for tracking
- **Query fields** to extract specific information from freeform documents (e.g., "What is the policy number?")
- **High-resolution mode** for faded faxes, poor quality scans, or fine print in policy documents

Document Intelligence supports these as optional **add-on capabilities** that you can enable per request.

---

## Prerequisites

Complete [00-setup-and-basics.ipynb](00-setup-and-basics.ipynb) first.

> **Note**: Some add-on capabilities incur additional charges. See [pricing](https://azure.microsoft.com/pricing/details/ai-document-intelligence/).

In [ ]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import (
    AnalyzeDocumentRequest,
    AnalyzeResult,
    DocumentAnalysisFeature,
)

load_dotenv(dotenv_path=os.path.join("..", ".env"))

_api_key = os.environ.get("DOCUMENTINTELLIGENCE_API_KEY", "")
_credential = AzureKeyCredential(_api_key) if _api_key else DefaultAzureCredential()

client = DocumentIntelligenceClient(
    endpoint=os.environ["DOCUMENTINTELLIGENCE_ENDPOINT"],
    credential=_credential
)
print("✅ Client ready")

## Available Add-on Features

| Feature | Enum Value | Use Case |
|---|---|---|
| Barcode/QR extraction | `BARCODES` | Read tracking barcodes on claim forms |
| Formula extraction | `FORMULAS` | Extract mathematical formulas (rare in insurance) |
| Font/style info | `STYLE_FONT` | Detect font changes, bold/italic (clause emphasis) |
| High resolution | `OCR_HIGH_RESOLUTION` | Better accuracy on faded faxes, small text |
| Language detection | `LANGUAGES` | Detect document language for multilingual support |
| Query fields | `QUERY_FIELDS` | Ask natural-language questions about document content |
| Key-value pairs | `KEY_VALUE_PAIRS` | Extract all key-value pairs from any document |

## Feature 1: Barcode & QR Code Extraction

**Insurance use case**: Claim forms often have barcodes for routing, tracking numbers on repair invoices, QR codes linking to digital claim portals.

In [ ]:
document_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-layout.pdf"

poller = client.begin_analyze_document(
    "prebuilt-layout",
    AnalyzeDocumentRequest(url_source=document_url),
    features=[DocumentAnalysisFeature.BARCODES]
)
result: AnalyzeResult = poller.result()

# Check for barcodes
for page in result.pages:
    if page.barcodes:
        print(f"Page {page.page_number} barcodes:")
        for barcode in page.barcodes:
            print(f"  Type: {barcode.kind}")
            print(f"  Value: {barcode.value}")
            print(f"  Confidence: {barcode.confidence:.2%}")
            print()
    else:
        print(f"Page {page.page_number}: No barcodes found")

print("\nTip: Test with a document containing barcodes (e.g., a shipping label or claim form).")

## Feature 2: Query Fields

**Insurance use case**: Extract specific pieces of information from freeform documents by asking natural-language questions. For example:
- "What is the policy number?" from a cover letter
- "What is the date of the accident?" from a police report
- "What is the total amount claimed?" from a handwritten claim note

> This is especially powerful for unstructured documents where no prebuilt model exists.

In [ ]:
document_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-layout.pdf"

# Ask specific questions about the document
poller = client.begin_analyze_document(
    "prebuilt-layout",
    AnalyzeDocumentRequest(url_source=document_url),
    features=[DocumentAnalysisFeature.QUERY_FIELDS],
    query_fields=["CompanyName", "Date", "TotalAmount", "Address"]
)
result: AnalyzeResult = poller.result()

print("QUERY FIELD RESULTS:")
print("=" * 50)
if result.documents:
    for doc in result.documents:
        if doc.fields:
            for name, field in doc.fields.items():
                value = field.get("content", "Not found")
                conf = field.get("confidence", 0)
                print(f"  {name:20s} → {str(value):30s} ({conf:.0%})")
else:
    print("No query field results returned.")

## Feature 3: High-Resolution Mode

**Insurance use case**: Faded faxes, old scanned documents, small-print policy terms — high-res mode helps extract text from low-quality sources.

In [ ]:
document_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-layout.pdf"

# Standard resolution
poller_standard = client.begin_analyze_document(
    "prebuilt-read",
    AnalyzeDocumentRequest(url_source=document_url),
)
result_standard = poller_standard.result()

# High resolution
poller_highres = client.begin_analyze_document(
    "prebuilt-read",
    AnalyzeDocumentRequest(url_source=document_url),
    features=[DocumentAnalysisFeature.OCR_HIGH_RESOLUTION]
)
result_highres = poller_highres.result()

# Compare
std_words = sum(len(p.words) for p in result_standard.pages if p.words)
hr_words = sum(len(p.words) for p in result_highres.pages if p.words)

print("RESOLUTION COMPARISON:")
print(f"  Standard: {std_words} words extracted")
print(f"  High-res: {hr_words} words extracted")
print(f"  Difference: {hr_words - std_words:+d} words")
print("\nTip: The difference is most noticeable on poor-quality scans and faxes.")

## Feature 4: Key-Value Pair Extraction

**Insurance use case**: Extract all key-value pairs from any document — useful for semi-structured forms where you don't know the field names in advance.

In [ ]:
document_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-layout.pdf"

poller = client.begin_analyze_document(
    "prebuilt-layout",
    AnalyzeDocumentRequest(url_source=document_url),
    features=[DocumentAnalysisFeature.KEY_VALUE_PAIRS]
)
result: AnalyzeResult = poller.result()

print("KEY-VALUE PAIRS:")
print("=" * 60)
if result.key_value_pairs:
    for kv in result.key_value_pairs:
        key = kv.key.content if kv.key else "N/A"
        value = kv.value.content if kv.value else "N/A"
        conf = kv.confidence
        print(f"  {key:30s} → {value:25s} ({conf:.0%})")
else:
    print("No key-value pairs found.")

## Combining Multiple Features

You can enable multiple features in a single request.

In [ ]:
poller = client.begin_analyze_document(
    "prebuilt-layout",
    AnalyzeDocumentRequest(url_source=document_url),
    features=[
        DocumentAnalysisFeature.BARCODES,
        DocumentAnalysisFeature.KEY_VALUE_PAIRS,
        DocumentAnalysisFeature.LANGUAGES,
    ]
)
result = poller.result()

print("Combined analysis:")
print(f"  Tables: {len(result.tables) if result.tables else 0}")
print(f"  Key-value pairs: {len(result.key_value_pairs) if result.key_value_pairs else 0}")

total_barcodes = sum(len(p.barcodes) for p in result.pages if p.barcodes)
print(f"  Barcodes: {total_barcodes}")

if result.languages:
    for lang in result.languages:
        print(f"  Language: {lang.locale} (confidence: {lang.confidence:.2%})")

---

## Key Takeaways

| Add-on | Insurance Value | Extra Cost? |
|---|---|---|
| **Barcodes** | Track claim forms, read shipment labels | Yes |
| **Query fields** | Ask questions about freeform documents (police reports, letters) | Yes |
| **High resolution** | Extract from faded faxes, small print in policy terms | Yes |
| **Key-value pairs** | Extract all form fields from unknown document types | Yes |
| **Language detection** | Route multilingual documents to the right team | No |

See [pricing](https://azure.microsoft.com/pricing/details/ai-document-intelligence/) for add-on costs.

## Next Steps

| Notebook | What You'll Learn |
|---|---|
| [10-human-in-the-loop.ipynb](10-human-in-the-loop.ipynb) | Build a confidence-based human review queue |
| [11-batch-processing.ipynb](11-batch-processing.ipynb) | Process documents at scale |